# ERCOT Products Endpoint Check

Pre-scraper review notebook, so human confirms endpoints, params, history depth, and expected file extraction time.

Setup: `cp .env.example .env`, fill ERCOT credentials, then Run All.

**Phase 1 — Learn:** endpoint URL, field schema, unfiltered sample.

**Phase 2 — History depth:** earliest date the live API returns.

**Phase 3 — Archive depth:** earliest doc in the archive API (backfill route).

**Phase 4 — Volume:** observed row counts → parquet size estimate.

In [1]:
import os, time
import requests
import pandas as pd
from datetime import date, timedelta
from dotenv import load_dotenv

load_dotenv()

_TOKEN_URL = (
    'https://ercotb2c.b2clogin.com'
    '/ercotb2c.onmicrosoft.com'
    '/B2C_1_PUBAPI-ROPC-FLOW'
    '/oauth2/v2.0/token'
)

_token_cache = {'token': None, 'expires_at': 0.0}

def _get_token() -> str:
    if _token_cache['token'] and time.time() < _token_cache['expires_at']:
        return _token_cache['token']
    resp = requests.post(_TOKEN_URL, data={
        'username'     : os.environ['ERCOT_USERNAME'],
        'password'     : os.environ['ERCOT_PASSWORD'],
        'grant_type'   : 'password',
        'client_id'    : 'fec253ea-0d06-4272-a5e6-b478baeecd70',
        'scope'        : f'openid fec253ea-0d06-4272-a5e6-b478baeecd70 offline_access',
        'response_type': 'id_token',
    }, timeout=(10, 60))
    resp.raise_for_status()
    body = resp.json()
    _token_cache['token']      = body['id_token']
    _token_cache['expires_at'] = time.time() + int(body.get('expires_in', 3600)) - 60
    return _token_cache['token']

def _hdrs() -> dict:
    return {
        'Authorization'            : f'Bearer {_get_token()}',
        'Ocp-Apim-Subscription-Key': os.environ['ERCOT_SUBSCRIPTION_KEY'],
        'Accept'                   : 'application/json',
    }

def _get(url, **kwargs):
    r = requests.get(url, headers=_hdrs(), timeout=(10, 120), **kwargs)
    r.raise_for_status()
    return r.json()

def _to_df(body: dict) -> pd.DataFrame:
    cols = [f['name'] if isinstance(f, dict) else f for f in body.get('fields', [])]
    return pd.DataFrame([dict(zip(cols, row)) for row in body.get('data', [])])

print('Auth OK —', _get_token()[:20], '...')

Auth OK — eyJhbGciOiJSUzI1NiIs ...


In [ ]:
# ── PARAMETERS ─────────────────────────────────────────────────────────────────
PRODUCTS = ['NP4-743-CD', 'NP4-746-CD']
API_BASE = 'https://api.ercot.com/api/public-reports'
# ───────────────────────────────────────────────────────────────────────────────

---
## Phase 1 — Endpoint & schema discovery

For each product: resolve the live endpoint from the product's artifact list, show every field
(`hasRange=True` ⇒ supports `XxxFrom`/`XxxTo` query params), and pull a 5-row unfiltered sample
(API defaults to today's window).

In [6]:
meta_body = _get(f'{API_BASE}/np4-743-cd')
artifact  = meta_body.get('artifacts', [])[0]
endpoint  = artifact['_links']['endpoint']['href']

        # sample data print and identify date field
probe = _get(endpoint, params={'page': 1, 'size': 5})
fields_df = pd.DataFrame([{
            'name'    : f.get('name'),
            'label'   : f.get('label'),
            'dataType': f.get('dataType'),
        } for f in meta_body.get('fields', [])])
print(meta_body)

{'emilId': 'NP4-743-CD', 'name': 'Wind Power Production - Actual 5-Minute Averaged Values by Geographical Region', 'description': 'This report is posted every 5 minutes and includes System-wide actual 5-minute averaged wind power production (GEN) and wind power potential (HSL) for a rolling historical 60-minute period. Actual 5-min averaged wind power production (GEN) is also included at the geographic regional level.', 'status': 'Active', 'reportTypeId': 14788, 'audience': 'Public', 'generationFrequency': 'Chron - 5 Minutes', 'securityClassification': 'Public', 'lastUpdated': '2024-10-29', 'firstRun': '2017-05-10', 'eceii': '', 'channel': 'Public,EWS,Data Portal', 'userGuide': None, 'postingType': 'Report', 'market': 'Nodal', 'extractSubscriber': '', 'xsdName': 'Current Day Reports XSD', 'misPostingLocation': '', 'certificateRole': '', 'fileType': 'zip,csv,xml', 'ddlName': '', 'misDisplayDuration': 7, 'archiveDuration': 2555, 'notificationType': '', 'contentType': 'DATA', 'downloadLim

In [3]:
from concurrent.futures import ThreadPoolExecutor

info = {}  # product -> {endpoint, fields_df, ts_field, min_date, max_date}

def _probe(product):
    try:
        meta_body = _get(f'{API_BASE}/{product.lower()}')
        artifact  = meta_body.get('artifacts', [])[0]
        endpoint  = artifact['_links']['endpoint']['href']

        # sample data print and identify date field
        probe = _get(endpoint, params={'page': 1, 'size': 5})
        fields_df = pd.DataFrame([{
            'name'    : f.get('name'),
            'label'   : f.get('label'),
            'dataType': f.get('dataType'),
        } for f in probe.get('fields', [])])

        ts_candidates = [n for n in fields_df['name']
                         if any(k in n.lower() for k in ('intervalending', 'deliverydate', 'operatingday'))]
        ts_field = ts_candidates[0] if ts_candidates else None
        
        sample_df=_to_df(probe)

        # min max date
        min_date=max_date=None
        if ts_field:
            asc = _get(endpoint, params={'sort':ts_field,'dir':'ASC','page':1,'size':1})
            desc = _get(endpoint, params={'sort':ts_field,'dir':'DESC','page':1,'size':1})
            asc_df, desc_df = _to_df(asc), _to_df(desc)
            min_date = asc_df[ts_field].iloc[0] if len(asc_df) else None
            max_date = desc_df[ts_field].iloc[0] if len(desc_df) else None
        
        return product, {
            'artifact' : artifact,
            'endpoint' : endpoint,
            'fields_df': fields_df,
            'ts_field' : ts_field,
            'min_date' : min_date,
            'max_date' : max_date,
            'sample_df': sample_df,
        }, None
    except requests.HTTPError as e:
        return product, None, e

# Products are independent -> fetch concurrently (I/O-bound calls, not CPU-bound).
with ThreadPoolExecutor(max_workers=min(8, len(PRODUCTS))) as ex:
    results = list(ex.map(_probe, PRODUCTS))

for product, result, err in results:
    print('=' * 100)
    if err is not None:
        print(f'{product}: HTTP error \u2014 {err}')
        continue

    artifact = result['artifact']
    print(f'{artifact["displayName"]}')
    print(f'EMIL ID        : {product}')
    print(f'Endpoint       : {result["endpoint"]}')
    print(f'Time Range     : {result["ts_field"]}  ({result["min_date"]} - {result["max_date"]})')
    display(result['sample_df'])

    info[product] = {
        'endpoint' : result['endpoint'],
        'fields_df': result['fields_df'],
        'ts_field' : result['ts_field'],
        'min_date' : result['min_date'],
        'max_date' : result['max_date'],
    }


Wind Power Production - Actual 5-Minute Averaged Values by Geographical Region
EMIL ID        : NP4-743-CD
Endpoint       : https://api.ercot.com/api/public-reports/np4-743-cd/wpp_actual_5min_avg_values_geo
Time Range     : intervalEnding  (2023-12-11T13:25:00 - 2026-07-08T15:50:00)


,postedDatetime,intervalEnding,genSystemWide,panhandle,coastal,south,west,north,HSLSystemWide,DSTFlag
0,2026-07-08T15:55:37,2026-07-08T15:50:00,11184.82,1823.16,1495.62,1366.95,6059.93,439.15,11528.17,False
1,2026-07-08T15:55:37,2026-07-08T15:45:00,11112.47,1828.11,1397.18,1356.75,6080.52,449.90,11452.53,False
2,2026-07-08T15:55:37,2026-07-08T15:40:00,10845.89,1798.45,1263.36,1350.99,5979.04,454.05,11183.26,False
3,2026-07-08T15:55:37,2026-07-08T15:35:00,10829.55,1811.30,1221.66,1329.81,6042.29,424.49,11160.66,False
4,2026-07-08T15:55:37,2026-07-08T15:30:00,10837.87,1822.79,1219.28,1304.32,6050.22,441.25,11168.10,False


NP4-746-CD: HTTP error — 429 Client Error: Too Many Requests for url: https://api.ercot.com/api/public-reports/np4-746-cd/spp_actual_5min_avg_values_geo?sort=intervalEnding&dir=ASC&page=1&size=1


---
## Phase 2 — Live-API history depth

Probe `totalRecords` for one-day windows at several past dates. First date with records > 0
≈ earliest live coverage. (Wind hourly NP4-742-CD only reached Dec 2023 live; expect similar here.)

In [10]:
PROBE_DATES = ['2021-01-15', '2022-01-15', '2023-01-15', '2023-12-15', '2024-06-15', '2025-06-15']

depth_rows = []
for product, d in info.items():
    ts = d['ts_field']
    if ts is None:
        continue
    for pd_date in PROBE_DATES:
        try:
            body = _get(d['endpoint'], params={
                f'{ts}From': pd_date, f'{ts}To': pd_date, 'page': 1, 'size': 1,
            })
            n = body.get('_meta', {}).get('totalRecords', 0)
        except requests.HTTPError as e:
            n = f'ERR {e.response.status_code}'
        depth_rows.append({'product': product, 'date': pd_date, 'totalRecords': n})
        time.sleep(0.3)

depth_df = pd.DataFrame(depth_rows).pivot(index='date', columns='product', values='totalRecords')
display(depth_df)
print('First row with records > 0 per column ≈ earliest live coverage.')

product,NP4-733-CD,NP4-738-CD,NP4-743-CD,NP4-745-CD,NP4-746-CD
date,,,,,
2021-01-15,0,ERR 429,0,ERR 429,0
2022-01-15,0,ERR 429,0,ERR 429,0
2023-01-15,0,ERR 429,0,ERR 429,0
2023-12-15,12,ERR 429,12,ERR 429,0
2024-06-15,12,ERR 429,12,ERR 429,0
2025-06-15,ERR 429,ERR 429,12,ERR 429,0


First row with records > 0 per column ≈ earliest live coverage.


---
## Phase 3 — Archive-API depth

The archive API (`/archive/{PRODUCT}`) is the backfill route if the live API is shallow
(same two-step docId → ZIP flow as `ercot_wpp_archive.py`). Listing is paginated newest-first;
the last page holds the oldest doc.

In [11]:
arch_rows = []
for product in info:
    url = f'{API_BASE}/archive/{product}'
    try:
        first = _get(url, params={'page': 1, 'size': 1})
        total = first.get('_meta', {}).get('totalRecords', 0)
        pages = first.get('_meta', {}).get('totalPages', 1)
        newest = first.get('archives', [{}])[0]
        oldest = _get(url, params={'page': pages, 'size': 1}).get('archives', [{}])[0]
        arch_rows.append({
            'product'    : product,
            'totalDocs'  : total,
            'newest_post': newest.get('postDatetime'),
            'oldest_post': oldest.get('postDatetime'),
        })
    except requests.HTTPError as e:
        arch_rows.append({'product': product, 'totalDocs': f'ERR {e.response.status_code}'})
    time.sleep(0.3)

display(pd.DataFrame(arch_rows))
print('oldest_post ≈ how far back a backfill can reach.')

,product,totalDocs,newest_post,oldest_post
0,NP4-743-CD,961930,2026-07-08T12:35:33.000,2017-05-11T14:55:32.532
1,NP4-746-CD,422946,2026-07-08T12:35:17.000,2022-06-30T16:45:27.349
2,NP4-733-CD,1281345,2026-07-08T12:35:31.000,2014-05-01T00:00:53.912
3,NP4-738-CD,1094577,2026-07-08T12:35:15.000,2016-02-10T00:00:07.519
4,NP4-745-CD,35246,2026-07-08T11:55:35.000,2022-06-30T16:55:28.984


oldest_post ≈ how far back a backfill can reach.


---
## Phase 4 — Volume & parquet size estimate

Observed `totalRecords` for one full recent month, then extrapolated to a 2021-01-01 → today pull.
Rows are **pre-dedupe** (each 5-min interval appears ~12× across rolling 60-min snapshots);
deduped rows ≈ raw ÷ 12. Bytes/row benchmarks from this repo:
~50 B (snappy, deduped hourly wind) → assume ~60 B for 5-min zstd, conservatively.

In [ ]:
MONTH_FROM, MONTH_TO = '2026-06-01', '2026-06-30'
BYTES_PER_ROW = 60
days_full = (date.today() - date(2021, 1, 1)).days

vol_rows = []
for product, d in info.items():
    ts = d['ts_field']
    if ts is None:
        continue
    try:
        body = _get(d['endpoint'], params={
            f'{ts}From': MONTH_FROM, f'{ts}To': MONTH_TO, 'page': 1, 'size': 1,
        })
        n_month = body.get('_meta', {}).get('totalRecords', 0)
    except requests.HTTPError:
        n_month = 0
    raw_full   = n_month / 30 * days_full
    dedup_full = raw_full / 12  # keep one snapshot per interval
    vol_rows.append({
        'product'          : product,
        'rows_june2026'    : n_month,
        'raw_rows_2021_now': f'{raw_full:,.0f}',
        'deduped_rows'     : f'{dedup_full:,.0f}',
        'est_parquet_MB'   : round(dedup_full * BYTES_PER_ROW / 1e6, 1),
    })
    time.sleep(0.3)

display(pd.DataFrame(vol_rows))

---
## Decision checklist (human)

- [ ] Wind product: NP4-743-CD (by geo) vs NP4-733-CD (system-wide)?
- [ ] Solar product: NP4-746-CD (by geo) vs NP4-738-CD (system-wide)?
- [ ] Confirm range-filter param names from Phase 1 (`ts_field`).
- [ ] Live depth sufficient (Phase 2), or archive backfill needed (Phase 3)?
- [ ] Estimated parquet size acceptable (Phase 4)?
- [ ] NP4-745-CD solar hourly: does live API really reach 2021? → run `ercot_spp_by_geo.py`.

Then: build `ercot_wpp_5min.py` / `ercot_spp_5min.py` (raw values kept, snapshot-deduped, zstd).